# Download RAGBench + embedder to Google Drive

Use this notebook on **Google Colab** to download phase-1.1 RAGBench subsets and the V1 embedder, then copy to your laptop.

**After download**, on your machine copy Drive folders to:

```
nextgen-rag-system/data/raw/ragbench/<subset>/train-00000-of-00001.parquet
nextgen-rag-system/models/bge-small-en-v1.5/
```

Then verify:

```bash
rag-kag data-status
rag-kag eda -o docs/eda/ragbench_eda.md --json experiments/eda/ragbench_stats.json
```

**Optional:** Colab → Secrets → add `HF_TOKEN` for higher HF rate limits.

In [ ]:
# Install / upgrade huggingface_hub (Colab usually has it; this keeps versions fresh)
!pip install -q -U huggingface_hub

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN", None)
except Exception:
    HF_TOKEN = None  # fine for public galileo-ai/ragbench

# --- paths on Google Drive ---
DRIVE_ROOT = Path("/content/drive/MyDrive/capstone-rag-kag")
RAGBENCH_DIR = DRIVE_ROOT / "data/raw/ragbench"
EMBEDDER_DIR = DRIVE_ROOT / "models/bge-small-en-v1.5"

RAGBENCH_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDER_DIR.mkdir(parents=True, exist_ok=True)

# Phase 1.1 representative subsets (one or two per domain)
PHASE1_SUBSETS = [
    "covidqa",
    "pubmedqa",
    "hotpotqa",
    "msmarco",
    "cuad",
    "emanual",
    "techqa",
    "finqa",
    "tatqa",
]

RAGBENCH_REPO = "galileo-ai/ragbench"
EMBEDDER_REPO = "BAAI/bge-small-en-v1.5"

# Download all splits (train + validation + test). Set False to fetch train only.
DOWNLOAD_ALL_SPLITS = True

print(f"RAGBench target: {RAGBENCH_DIR}")
print(f"Embedder target: {EMBEDDER_DIR}")
print(f"HF_TOKEN set: {HF_TOKEN is not None}")

In [ ]:
from huggingface_hub import snapshot_download

results = []

for subset in PHASE1_SUBSETS:
    if DOWNLOAD_ALL_SPLITS:
        patterns = [f"{subset}/*"]
    else:
        patterns = [f"{subset}/train-*"]

    print(f"\n=== Downloading {subset} ===")
    try:
        path = snapshot_download(
            repo_id=RAGBENCH_REPO,
            repo_type="dataset",
            allow_patterns=patterns,
            local_dir=str(RAGBENCH_DIR),
            local_dir_use_symlinks=False,
            token=HF_TOKEN,
        )
        subset_path = RAGBENCH_DIR / subset
        parquets = sorted(subset_path.glob("*.parquet")) if subset_path.is_dir() else []
        results.append((subset, "ok", len(parquets), str(subset_path)))
        print(f"  -> {len(parquets)} parquet file(s) in {subset_path}")
    except Exception as exc:
        results.append((subset, "FAIL", 0, str(exc)))
        print(f"  -> ERROR: {exc}")

print("\n--- RAGBench summary ---")
for subset, status, n_files, detail in results:
    print(f"{subset:10} {status:4}  files={n_files}  {detail}")

In [ ]:
# V1 embedder weights (same layout as configs/v1_baseline.yaml)
from huggingface_hub import snapshot_download

print(f"Downloading embedder {EMBEDDER_REPO} ...")
snapshot_download(
    repo_id=EMBEDDER_REPO,
    local_dir=str(EMBEDDER_DIR),
    local_dir_use_symlinks=False,
    token=HF_TOKEN,
)
print(f"Embedder saved to {EMBEDDER_DIR}")

In [ ]:
# Verify layout matches nextgen-rag-system expectations
from pathlib import Path

print("RAGBench parquet files:")
for subset in PHASE1_SUBSETS:
    subset_dir = RAGBENCH_DIR / subset
    if not subset_dir.is_dir():
        print(f"  MISSING folder: {subset_dir}")
        continue
    for pq in sorted(subset_dir.glob("*.parquet")):
        size_mb = pq.stat().st_size / (1024 * 1024)
        print(f"  {pq.relative_to(RAGBENCH_DIR)}  ({size_mb:.2f} MB)")

embedder_files = list(EMBEDDER_DIR.rglob("*"))
print(f"\nEmbedder: {len(embedder_files)} files under {EMBEDDER_DIR}")

print("\n--- Copy to laptop ---")
print(f"1. Download folder: {DRIVE_ROOT}")
print("2. Merge into repo:")
print(f"   cp -R '{RAGBENCH_DIR}'/*  <repo>/data/raw/ragbench/")
print(f"   cp -R '{EMBEDDER_DIR}'     <repo>/models/bge-small-en-v1.5/")